In [ ]:
# Enterprise FinTech Payment Intelligence Platform
## Phase 4 – Machine Learning
### Notebook 03 – Feature Engineering

**Objective:**
Transform the cleaned analytical dataset into a machine learning-ready dataset by creating meaningful features that help identify fraudulent transaction behavior.

In [3]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("clean_ml_dataset.csv")
print("Dataset Loaded Successfully")
print(f"Original Shape: {df.shape}")
df.head()

Dataset Loaded Successfully
Original Shape: (6362620, 14)


,TransactionID,DayNumber,HourOfSimulation,PeriodOfDay,TransactionType,SourceAccountID,DestinationAccountID,IsFraud,IsFlaggedFraud,Amount,OldBalanceOrig,NewBalanceOrig,OldBalanceDest,NewBalanceDest
0,20666,1,13,Afternoon,PAYMENT,C2108058134,M1963634359,False,False,22005.03,0.00,0.00,0.00,0.00
1,20851,1,15,Afternoon,CASH_IN,C1672847392,C882180306,False,False,214524.46,5030.00,219554.46,0.00,0.00
2,21169,1,15,Afternoon,TRANSFER,C519692057,C834458122,False,False,1432648.47,0.00,0.00,1453236.68,2885885.15
3,21181,1,15,Afternoon,CASH_OUT,C1634465843,C807329566,False,False,352807.10,0.00,0.00,426401.78,779208.88
4,21321,1,15,Afternoon,CASH_OUT,C210125456,C410910993,False,False,275348.64,390959.31,115610.67,0.00,275348.64


In [5]:
# Enterprise Best Practice: Never modify the original dataframe in place.
df_fe = df.copy()

In [7]:
# Business Meaning: How much money actually left the sender's account.
df_fe["OriginBalanceChange"] = (
    df_fe["OldBalanceOrig"] - df_fe["NewBalanceOrig"]
)

In [8]:
# Business Meaning: How much money was actually received by the destination account.
df_fe["DestinationBalanceChange"] = (
    df_fe["NewBalanceDest"] - df_fe["OldBalanceDest"]
)

In [9]:
# Business Meaning: The percentage of the sender's total balance being transferred.
# (+1 avoids divide-by-zero errors for zero-balance accounts)
df_fe["AmountToOriginBalanceRatio"] = (
    df_fe["Amount"] / (df_fe["OldBalanceOrig"] + 1)
)

In [10]:
# Business Meaning: Flags outlier transactions sitting in the top 5% of all payment volume.
threshold = df_fe["Amount"].quantile(0.95)

df_fe["HighValueTransaction"] = (
    df_fe["Amount"] >= threshold
).astype(int)

In [11]:
# Business Meaning: Captures the wealth disparity between the sender and receiver.
df_fe["BalanceGap"] = abs(
    df_fe["OldBalanceOrig"] - df_fe["OldBalanceDest"]
)

In [17]:
# Convert one-hot encoded boolean columns to integers (0/1)
dummy_cols = [
    col for col in df_fe.columns
    if col.startswith("PeriodOfDay_") or col.startswith("TransactionType_")
]

df_fe[dummy_cols] = df_fe[dummy_cols].astype(int)

In [18]:
# Ensure target and flag columns are strictly integers (0 and 1)
df_fe["IsFraud"] = df_fe["IsFraud"].astype(int)
df_fe["IsFlaggedFraud"] = df_fe["IsFlaggedFraud"].astype(int)

In [21]:
print(f"Engineered Shape: {df_fe.shape}")
df_fe.head()

Engineered Shape: (6362620, 24)


,TransactionID,DayNumber,HourOfSimulation,SourceAccountID,DestinationAccountID,IsFraud,IsFlaggedFraud,Amount,OldBalanceOrig,NewBalanceOrig,...,AmountToOriginBalanceRatio,HighValueTransaction,BalanceGap,PeriodOfDay_Evening,PeriodOfDay_Morning,PeriodOfDay_Night,TransactionType_CASH_OUT,TransactionType_DEBIT,TransactionType_PAYMENT,TransactionType_TRANSFER
0,20666,1,13,C2108058134,M1963634359,0,0,22005.03,0.00,0.00,...,2.200503e+04,0,0.00,0,0,0,0,0,1,0
1,20851,1,15,C1672847392,C882180306,0,0,214524.46,5030.00,219554.46,...,4.264052e+01,0,5030.00,0,0,0,0,0,0,0
2,21169,1,15,C519692057,C834458122,0,0,1432648.47,0.00,0.00,...,1.432648e+06,1,1453236.68,0,0,0,0,0,0,1
3,21181,1,15,C1634465843,C807329566,0,0,352807.10,0.00,0.00,...,3.528071e+05,0,426401.78,0,0,0,1,0,0,0
4,21321,1,15,C210125456,C410910993,0,0,275348.64,390959.31,115610.67,...,7.042880e-01,0,390959.31,0,0,0,1,0,0,0


In [22]:
print("Checking for missing values created during feature engineering:")
print(df_fe.isnull().sum())

Checking for missing values created during feature engineering:
TransactionID                 0
DayNumber                     0
HourOfSimulation              0
SourceAccountID               0
DestinationAccountID          0
IsFraud                       0
IsFlaggedFraud                0
Amount                        0
OldBalanceOrig                0
NewBalanceOrig                0
OldBalanceDest                0
NewBalanceDest                0
OriginBalanceChange           0
DestinationBalanceChange      0
AmountToOriginBalanceRatio    0
HighValueTransaction          0
BalanceGap                    0
PeriodOfDay_Evening           0
PeriodOfDay_Morning           0
PeriodOfDay_Night             0
TransactionType_CASH_OUT      0
TransactionType_DEBIT         0
TransactionType_PAYMENT       0
TransactionType_TRANSFER      0
dtype: int64


In [23]:
output_filename = "feature_engineered_dataset.csv"

df_fe.to_csv(output_filename, index=False)
print(f"Feature Engineering Completed Successfully. Saved to {output_filename}")

Feature Engineering Completed Successfully. Saved to feature_engineered_dataset.csv


In [ ]:
### 💡 Feature Engineering Summary

* **Balance Movements:** Created sender and receiver balance movement features to track physical money flow.
* **Financial Behavior:** Derived transaction-to-balance ratio to capture aggressive account draining.
* **Risk Thresholds:** Identified high-value transactions using a dynamic 95th percentile threshold.
* **Account Disparity:** Calculated the balance gap between sender and receiver.
* **ML Formatting:** Encoded categorical variables and standardized booleans for model compatibility.
* **Data Integrity:** Verified zero missing values post-engineering.
* **Export:** Exported the final feature-engineered dataset for preprocessing and model training.